# Building an NBA squad under the salary cap: stats first, salaries as the constraint

**Question.** Using only box-score based player ratings and contract data, can one assemble a 15-man squad that fits under the salary cap and is clearly stronger than real NBA teams? And how much of that is real, and how much is selection luck?

**Design (fixed before looking at results).**
- Squad chosen with the **previous** season's ratings (BPM, season N) and the **next** season's contracts (N+1). It is evaluated with players' **actual** ratings in season N+1.
- Rotation of 10 players, two per position, plus 5 depth places at near-minimum salary (they carry salary, but no minutes).
- Constraints: salary cap of season N+1; at most 2 all-stars, at most 3 rookie-scale contracts, at most 2 max-like contracts (25% of the cap or more); players need 1,000+ minutes in season N.
- Team strength = sum of BPM x minutes / 48 (a minutes-weighted BPM sum). An optional usage constraint keeps the minutes-weighted usage of the squad at or below 100, as in a real lineup.
- Two season pairs: 2024 -> 2025 and 2025 -> 2026 (seasons are named by the year they end). Earlier salaries in the open dataset did not match known contracts, so they were not used.

## 0. Data and setup

Files are not redistributed here. Download them and put them into a Google Drive folder named `nba_portfolio`:

- From Kaggle `sumitrodatta/nba-aba-baa-stats`: `Advanced.csv`, `Player Season Info.csv`, `All-Star Selections.csv`, `Draft Pick History.csv`
- From Kaggle `deocheng/nba-data-1946-2026`: `public.player_salaries_historical.csv`, `public.player_contracts.csv`

In [1]:
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/nba_portfolio"

Mounted at /content/drive


## 1. Quick data check

Seasons, number of players with 1,000+ minutes, positions, and a sanity check of the top BPM values.

In [2]:
import pandas as pd

folder = "/content/drive/MyDrive/nba_portfolio"
adv = pd.read_csv(f"{folder}/Advanced.csv")

# NBA only, seasons 2022-2026 (season = year the season ends)
d = adv[(adv["lg"] == "NBA") & (adv["season"].between(2022, 2026))].copy()

# traded players: keep the combined TOT row when it exists
has_tot = d.groupby(["season", "player_id"])["team"].transform(lambda s: (s == "TOT").any())
d = d[(~has_tot) | (d["team"] == "TOT")].copy()

# main position: first label of entries like "PG-SG"
d["pos1"] = d["pos"].astype(str).str.split("-").str[0]

print("Rows after cleaning:", len(d))
print()
print("Season | players | max games | players with 1000+ minutes")
for s, g in d.groupby("season"):
    print(s, "|", len(g), "|", g["g"].max(), "|", (g["mp"] >= 1000).sum())

print()
print("Positions of players with 1000+ minutes (season 2026):")
print(d[(d["season"] == 2026) & (d["mp"] >= 1000)]["pos1"].value_counts())

print()
print("Top 10 by BPM, season 2026, 1000+ minutes:")
cols = ["player", "team", "pos1", "age", "mp", "bpm", "vorp", "ws_48"]
top = d[(d["season"] == 2026) & (d["mp"] >= 1000)].sort_values("bpm", ascending=False)
print(top[cols].head(10).to_string(index=False))

Rows after cleaning: 3694

Season | players | max games | players with 1000+ minutes
2022 | 812 | 82 | 291
2023 | 679 | 83 | 281
2024 | 735 | 84 | 286
2025 | 735 | 82 | 283
2026 | 733 | 82 | 292

Positions of players with 1000+ minutes (season 2026):
pos1
SG    77
SF    60
C     55
PF    50
PG    50
Name: count, dtype: int64

Top 10 by BPM, season 2026, 1000+ minutes:
                 player team pos1  age     mp  bpm  vorp  ws_48
           Nikola Jokić  DEN    C 30.0 2265.0 14.2   9.2  0.316
Shai Gilgeous-Alexander  OKC   PG 27.0 2259.0 11.7   7.8  0.323
      Victor Wembanyama  SAS    C 22.0 1866.0 10.7   6.0  0.257
  Giannis Antetokounmpo  MIL   PF 31.0 1039.0  9.5   3.0  0.231
            Luka Dončić  LAL   PG 26.0 2289.0  9.3   6.6  0.199
          Kawhi Leonard  LAC   SF 34.0 2085.0  8.0   5.3  0.212
        Cade Cunningham  DET   PG 24.0 2172.0  6.3   4.6  0.174
           Jimmy Butler  GSW   SF 36.0 1182.0  5.5   2.3  0.233
          Stephen Curry  GSW   PG 37.0 1329.0  5.4   

## 2. Player pools for each season pair

For each pair (N, N+1): ratings of season N, contract of season N+1, outcome in season N+1, plus flags for all-stars, rookie-scale contracts (first-round pick in the first four seasons, an approximation) and max-like contracts. Players without a known salary, and two-way contracts, are excluded.

In [3]:
import re
import unicodedata
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/nba_portfolio"
adv = pd.read_csv(f"{folder}/Advanced.csv")
info = pd.read_csv(f"{folder}/Player Season Info.csv")
allstar = pd.read_csv(f"{folder}/All-Star Selections.csv")
draft = pd.read_csv(f"{folder}/Draft Pick History.csv")
sal = pd.read_csv(f"{folder}/public.player_salaries_historical.csv")
con = pd.read_csv(f"{folder}/public.player_contracts.csv")

# salary cap by season (2026 from the rules file; 2025 entered by hand)
CAP = {2025: 140588000, 2026: 154647000}


def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z ]", " ", s)
    return " ".join(s.split())


def one_row(df):
    g = df.groupby(["season", "player_id"])["team"]
    has_tot = g.transform(lambda s: (s == "TOT").any())
    df = df[(~has_tot) | (df["team"] == "TOT")]
    return df.drop_duplicates(["season", "player_id"])


a = one_row(adv[adv["lg"] == "NBA"].copy())
a["pos1"] = a["pos"].astype(str).str.split("-").str[0]
a["key"] = a["player"].map(norm)

i = one_row(info[info["lg"] == "NBA"].copy())
i = i[["season", "player_id", "experience"]]

rnd = pd.to_numeric(draft["round"], errors="coerce")
r1 = set(draft.loc[rnd == 1, "player_id"].dropna())

tw = con["is_two_way"].fillna(False)
tw = tw.astype(str).str.lower().isin(["true", "1"])
con = con.assign(two_way=tw).drop_duplicates("player_id")
con = con.set_index("player_id")

sal["key"] = sal["player_name"].map(norm)

BASE_COLS = ["player_id", "player", "key", "age", "pos1",
             "mp", "bpm", "vorp", "ws_48"]
NEXT_COLS = ["player_id", "mp", "bpm", "vorp", "ws_48"]


def build(N):
    M = N + 1
    cur = a[(a["season"] == N) & (a["mp"] >= 1000)]
    base = cur[BASE_COLS].copy()
    base = base.rename(columns={"mp": "mp_N", "bpm": "bpm_N",
                                "vorp": "vorp_N", "ws_48": "ws48_N"})

    nxt = a[a["season"] == M][NEXT_COLS]
    nxt = nxt.rename(columns={"mp": "mp_next", "bpm": "bpm_next",
                              "vorp": "vorp_next", "ws_48": "ws48_next"})
    base = base.merge(nxt, on="player_id", how="left")

    if M <= 2025:
        s1 = sal[sal["season"] == M].groupby("key")["salary"].max()
        base["salary_next"] = base["key"].map(s1)
        base["two_way_next"] = False
    else:
        base["salary_next"] = base["player_id"].map(con["salary_2025_26"])
        twm = base["player_id"].map(con["two_way"])
        base["two_way_next"] = twm.fillna(False)

    expN = i[i["season"] == N].set_index("player_id")["experience"]
    ex = pd.to_numeric(base["player_id"].map(expN), errors="coerce")
    base["exp_next"] = ex + 1
    is_r1 = base["player_id"].isin(r1)
    base["rookie_scale"] = is_r1 & (base["exp_next"] <= 4)

    stars = allstar.loc[allstar["season"] == N, "player_id"]
    base["allstar_N"] = base["player_id"].isin(set(stars))
    base["max_like"] = base["salary_next"] >= 0.25 * CAP[M]
    base["season_N"] = N
    base["season_next"] = M
    return base


pools = {}
for N in (2024, 2025):
    p = build(N)
    pools[N] = p
    p.to_csv(f"{folder}/pool_{N}_{N + 1}.csv", index=False)
    print("=== Pair", N, "->", N + 1, "===")
    print("players with 1000+ min:", len(p))
    print("share with salary:", round(p["salary_next"].notna().mean(), 3))
    print("share with BPM next season:", round(p["bpm_next"].notna().mean(), 3))
    print("all-stars in season N:", int(p["allstar_N"].sum()))
    print("rookie-scale contracts:", int(p["rookie_scale"].sum()))
    print("max-like contracts:", int(p["max_like"].sum()))
    print("positions:", p["pos1"].value_counts().to_dict())
    print("without salary, examples:")
    miss = p[p["salary_next"].isna()]
    print(miss[["player", "pos1", "mp_N", "bpm_N"]].head(10))
    print()

=== Pair 2024 -> 2025 ===
players with 1000+ min: 266
share with salary: 0.985
share with BPM next season: 0.977
all-stars in season N: 26
rookie-scale contracts: 52
max-like contracts: 34
positions: {'SG': 58, 'SF': 58, 'PG': 54, 'PF': 52, 'C': 44}
without salary, examples:
               player pos1    mp_N  bpm_N
24   Patrick Beverley   PG  1462.0   -1.1
107    Gordon Hayward   SF  1245.0   -2.1
185        Cedi Osman   SF  1269.0   -2.1
219  Dennis Smith Jr.   PG  1059.0   -0.8



/tmp/ipykernel_8003/1394157960.py:73: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base["two_way_next"] = twm.fillna(False)


=== Pair 2025 -> 2026 ===
players with 1000+ min: 270
share with salary: 0.963
share with BPM next season: 0.952
all-stars in season N: 26
rookie-scale contracts: 55
max-like contracts: 31
positions: {'SG': 70, 'SF': 53, 'PF': 53, 'PG': 49, 'C': 45}
without salary, examples:
                player pos1    mp_N  bpm_N
24       Malik Beasley   SG  2283.0    0.8
35        Jared Butler   SG  1044.0   -1.0
57    Ricky Council IV   SF  1250.0   -4.3
67   Spencer Dinwiddie   PG  2136.0   -1.6
126    DaQuan Jeffries   SG  1071.0   -4.7
137      Johnny Juzang   SG  1270.0   -2.3
153         Trey Lyles   PF  1349.0   -0.7
158        Cody Martin   SF  1173.0   -1.7
159      Tyrese Martin   SG  1315.0   -2.6
223        Ben Simmons   PG  1120.0   -0.5



## 3. Optimizer: base case (36 minutes for starters, 12 for the bench)

Mixed-integer program. In each position the better of the two chosen players (by season-N BPM) is the starter. Results are shown without and with the usage constraint. "Expected" uses season-N ratings, "got" uses season-N+1 ratings.

In [4]:
from scipy.optimize import milp, LinearConstraint, Bounds

POS = ["PG", "SG", "SF", "PF", "C"]
W_START, W_BENCH = 36, 12


def depth_cost(M):
    if M <= 2025:
        s = sal.loc[sal["season"] == M, "salary"]
    else:
        s = con.loc[~con["two_way"], "salary_2025_26"]
    s = s[s > 500000]
    return 5 * float(s.quantile(0.10))


def real_team_strength(M):
    t = adv[(adv["lg"] == "NBA") & (adv["season"] == M)]
    t = t[t["team"] != "TOT"].dropna(subset=["bpm", "mp"])
    t = t.assign(w=t["bpm"] * t["mp"])
    g = t.groupby("team")
    return 5 * g["w"].sum() / g["mp"].sum()


def prep(N):
    p = pools[N].copy()
    p = p[p["salary_next"].notna()]
    p = p[~p["two_way_next"].astype(bool)]
    usg = a[a["season"] == N].set_index("player_id")["usg_percent"]
    p["usg"] = p["player_id"].map(usg)
    p = p.dropna(subset=["bpm_N", "usg"])
    p["mp_next"] = p["mp_next"].fillna(0)
    ok = p["mp_next"] >= 500
    p["bpm_real"] = np.where(ok, p["bpm_next"], -2.0)
    for c in ["allstar_N", "rookie_scale", "max_like"]:
        p[c] = p[c].astype(int)
    return p.reset_index(drop=True)


def solve(N, use_usage, budget):
    M = N + 1
    p = prep(N)
    dc = depth_cost(M)
    n = len(p)
    bpm = p["bpm_N"].to_numpy(float)
    sl = p["salary_next"].to_numpy(float)
    usg = p["usg"].to_numpy(float)
    c = -np.concatenate([bpm * W_START, bpm * W_BENCH]) / 48

    rows = []
    lo = []
    hi = []
    zero = np.zeros(n)

    def add(cs, cb, low, high):
        rows.append(np.concatenate([cs, cb]))
        lo.append(low)
        hi.append(high)

    for k in range(n):
        e = np.zeros(n)
        e[k] = 1
        add(e, e, -np.inf, 1)
    for ps in POS:
        m = (p["pos1"] == ps).to_numpy(float)
        add(m, zero, 1, 1)
        add(zero, m, 1, 1)
    add(sl, sl, -np.inf, budget - dc)
    limits = [("allstar_N", 2), ("rookie_scale", 3), ("max_like", 2)]
    for col, lim in limits:
        v = p[col].to_numpy(float)
        add(v, v, -np.inf, lim)
    if use_usage:
        add(usg * W_START, usg * W_BENCH, -np.inf, 100 * 48)

    A = np.vstack(rows)
    cons = LinearConstraint(A, np.array(lo), np.array(hi))
    res = milp(c, constraints=cons, integrality=np.ones(2 * n),
               bounds=Bounds(0, 1))
    if res.x is None:
        print("no solution:", res.message)
        return None

    x = np.round(res.x).astype(int)
    st = np.where(x[:n] == 1)[0]
    bn = np.where(x[n:] == 1)[0]
    t1 = p.loc[st].assign(role="start", min=W_START)
    t2 = p.loc[bn].assign(role="bench", min=W_BENCH)
    t = pd.concat([t1, t2])
    return {
        "team": t,
        "pred": float((t["bpm_N"] * t["min"]).sum() / 48),
        "real": float((t["bpm_real"] * t["min"]).sum() / 48),
        "payroll": float(t["salary_next"].sum() + dc),
        "usage": float((t["usg"] * t["min"]).sum() / 48),
        "depth": dc,
    }


for N in (2024, 2025):
    M = N + 1
    rt = real_team_strength(M)
    print("=== Squad from season", N, ", checked on", M, "===")
    print("real teams: median", round(rt.median(), 1),
          ", best", round(rt.max(), 1))
    for use_usage in (False, True):
        r = solve(N, use_usage, CAP[M])
        if r is None:
            continue
        print("--- usage limit:", use_usage)
        print("expected (BPM of season N):", round(r["pred"], 1))
        print("got (BPM of season M):", round(r["real"], 1))
        print("payroll, M:", round(r["payroll"] / 1e6, 1),
              "of", round(CAP[M] / 1e6, 1),
              "(depth:", round(r["depth"] / 1e6, 1), ")")
        print("usage sum:", round(r["usage"], 1))
        cols = ["player", "pos1", "role", "salary_next",
                "bpm_N", "bpm_real", "usg"]
        t = r["team"].sort_values(["role", "pos1"], ascending=[False, True])
        print(t[cols].to_string(index=False))
        print()

=== Squad from season 2024 , checked on 2025 ===
real teams: median -0.2 , best 13.9
--- usage limit: False
expected (BPM of season N): 26.2
got (BPM of season M): 20.3
payroll, M: 138.9 of 140.6 (depth: 9.5 )
usage sum: 116.1
              player pos1  role  salary_next  bpm_N  bpm_real  usg
        Nikola Jokić    C start   51415938.0   13.2      13.3 29.3
Trayce Jackson-Davis   PF start    1891857.0    2.8       1.2 16.4
         Luka Dončić   PG start   43031940.0    9.9       6.7 36.0
     Trey Murphy III   SF start    5159854.0    2.5       0.8 18.6
    Donte DiVincenzo   SG start   11445000.0    3.4       1.9 21.3
      Alperen Şengün    C bench    5424654.0    4.9       4.4 27.0
      Jalen Williams   PF bench    4775760.0    2.3       4.0 23.7
          Tyus Jones   PG bench    2087519.0    0.7       0.1 16.3
          Sam Hauser   SF bench    2092344.0    1.2       0.8 15.0
         Sam Merrill   SG bench    2164993.0    0.5       0.3 17.5

--- usage limit: True
expected (BPM

## 4. Baseline: random squads under the same rules

Random squads that satisfy the same position, budget and category rules, compared with real teams evaluated with the same minutes template.

In [5]:
def real_team_template(M):
    t = adv[(adv["lg"] == "NBA") & (adv["season"] == M)]
    t = t[t["team"] != "TOT"].dropna(subset=["bpm", "mp"])
    wts = np.array([W_START] * 5 + [W_BENCH] * 5)
    out = {}
    for tm, g in t.groupby("team"):
        g = g.sort_values("mp", ascending=False).head(10)
        w = wts[: len(g)]
        out[tm] = float((g["bpm"].to_numpy() * w).sum() / 48)
    return pd.Series(out)


def random_teams(N, k, use_usage, seed=0):
    rng = np.random.default_rng(seed)
    M = N + 1
    p = prep(N)
    budget = CAP[M] - depth_cost(M)
    by_pos = {ps: np.where(p["pos1"] == ps)[0] for ps in POS}
    sl = p["salary_next"].to_numpy(float)
    real = p["bpm_real"].to_numpy(float)
    bN = p["bpm_N"].to_numpy(float)
    usg = p["usg"].to_numpy(float)
    star = p["allstar_N"].to_numpy()
    rook = p["rookie_scale"].to_numpy()
    mx = p["max_like"].to_numpy()
    res = []
    tries = 0
    while len(res) < k and tries < 300000:
        tries += 1
        idx = []
        for ps in POS:
            idx += list(rng.choice(by_pos[ps], 2, replace=False))
        idx = np.array(idx)
        if sl[idx].sum() > budget:
            continue
        if star[idx].sum() > 2 or rook[idx].sum() > 3 or mx[idx].sum() > 2:
            continue
        tot = 0.0
        u = 0.0
        for q in range(0, 10, 2):
            i1 = idx[q]
            i2 = idx[q + 1]
            if bN[i1] < bN[i2]:
                i1, i2 = i2, i1
            tot += real[i1] * W_START + real[i2] * W_BENCH
            u += usg[i1] * W_START + usg[i2] * W_BENCH
        if use_usage and u / 48 > 100:
            continue
        res.append(tot / 48)
    return res


for N in (2024, 2025):
    M = N + 1
    rt = real_team_template(M)
    print("=== pair", N, "->", M, "===")
    print("real teams, same minutes template: median",
          round(rt.median(), 1), ", best", round(rt.max(), 1))
    for use_usage in (False, True):
        rnd = np.array(random_teams(N, 2000, use_usage))
        r = solve(N, use_usage, CAP[M])
        print("usage:", use_usage)
        print("  random squads: accepted", len(rnd),
              ", median", round(float(np.median(rnd)), 1),
              ", 90th percentile", round(float(np.percentile(rnd, 90)), 1))
        print("  optimizer:", round(r["real"], 1),
              ", share of random squads below it:",
              round(float((rnd < r["real"]).mean()), 3))
    print()

=== pair 2024 -> 2025 ===
real teams, same minutes template: median -0.1 , best 16.2
usage: False
  random squads: accepted 2000 , median 1.0 , 90th percentile 6.1
  optimizer: 20.3 , share of random squads below it: 1.0
usage: True
  random squads: accepted 2000 , median 0.7 , 90th percentile 5.6
  optimizer: 24.1 , share of random squads below it: 1.0

=== pair 2025 -> 2026 ===
real teams, same minutes template: median 2.8 , best 15.6
usage: False
  random squads: accepted 2000 , median -0.7 , 90th percentile 4.7
  optimizer: 19.3 , share of random squads below it: 1.0
usage: True
  random squads: accepted 2000 , median -1.2 , 90th percentile 3.8
  optimizer: 22.3 , share of random squads below it: 1.0



## 5. Stronger baseline: greedy selection, and the optimizer without the top stars

Greedy selection takes the best BPM players one by one while budget and rules allow it. The last part removes the top one or two players by BPM from the pool to check how much of the result depends on them.

In [6]:
def greedy_team(N):
    M = N + 1
    p = prep(N)
    budget = CAP[M] - depth_cost(M)
    plain = p[(p["rookie_scale"] == 0) & (p["allstar_N"] == 0)
              & (p["max_like"] == 0)]
    cheap = {}
    for ps in POS:
        s = plain.loc[plain["pos1"] == ps, "salary_next"]
        cheap[ps] = sorted(s.tolist())

    def floor_cost(cnt):
        tot = 0.0
        for ps in POS:
            tot += sum(cheap[ps][: 2 - cnt[ps]])
        return tot

    order = p.sort_values("bpm_N", ascending=False).index
    cnt = {ps: 0 for ps in POS}
    chosen = []
    spent = 0.0
    c_star = 0
    c_rook = 0
    c_max = 0
    for k in order:
        if len(chosen) == 10:
            break
        ps = p.at[k, "pos1"]
        if cnt[ps] >= 2:
            continue
        if c_star + p.at[k, "allstar_N"] > 2:
            continue
        if c_rook + p.at[k, "rookie_scale"] > 3:
            continue
        if c_max + p.at[k, "max_like"] > 2:
            continue
        new_cnt = dict(cnt)
        new_cnt[ps] += 1
        need = spent + p.at[k, "salary_next"] + floor_cost(new_cnt)
        if need > budget:
            continue
        chosen.append(k)
        cnt = new_cnt
        spent += p.at[k, "salary_next"]
        c_star += p.at[k, "allstar_N"]
        c_rook += p.at[k, "rookie_scale"]
        c_max += p.at[k, "max_like"]
    if len(chosen) < 10:
        print("greedy selection filled only", len(chosen), cnt)
        return None
    pred = 0.0
    real = 0.0
    use = 0.0
    for ps in POS:
        ids = [k for k in chosen if p.at[k, "pos1"] == ps]
        ids.sort(key=lambda k: -p.at[k, "bpm_N"])
        for k, w in zip(ids, [W_START, W_BENCH]):
            pred += p.at[k, "bpm_N"] * w
            real += p.at[k, "bpm_real"] * w
            use += p.at[k, "usg"] * w
    return {"pred": pred / 48, "real": real / 48, "usage": use / 48}


for N in (2024, 2025):
    M = N + 1
    print("=== pair", N, "->", M, "===")
    g = greedy_team(N)
    r0 = solve(N, False, CAP[M])
    r1 = solve(N, True, CAP[M])
    if g is not None:
        print("greedy selection: expected", round(g["pred"], 1),
              ", got", round(g["real"], 1),
              ", usage", round(g["usage"], 1))
    print("optimizer without usage limit: got", round(r0["real"], 1))
    print("optimizer with usage limit: got", round(r1["real"], 1))

    saved = pools[N]
    top = saved.sort_values("bpm_N", ascending=False)["player"].tolist()
    for drop in (1, 2):
        names = top[:drop]
        pools[N] = saved[~saved["player"].isin(names)]
        r = solve(N, True, CAP[M])
        pools[N] = saved
        print("without", names, "(usage limit): got", round(r["real"], 1))
    print()

=== pair 2024 -> 2025 ===
greedy selection: expected 20.0 , got 14.0 , usage 103.8
optimizer without usage limit: got 20.3
optimizer with usage limit: got 24.1
without ['Nikola Jokić'] (usage limit): got 22.0
without ['Nikola Jokić', 'Joel Embiid'] (usage limit): got 22.0

=== pair 2025 -> 2026 ===
greedy selection: expected 22.3 , got 15.6 , usage 110.3
optimizer without usage limit: got 19.3
optimizer with usage limit: got 22.3
without ['Nikola Jokić'] (usage limit): got 18.3
without ['Nikola Jokić', 'Shai Gilgeous-Alexander'] (usage limit): got 15.9



## 6. Other minutes templates and a larger budget

Five minutes templates (36/12, 34/14, 32/16, a descending ladder, and 30/14/4 per position with 15 players), and a budget up to the luxury tax line instead of the cap. The first check line should reproduce the base case.

In [7]:
TAX = {2025: 170814000, 2026: 187895000}


def solve_gen(N, use_usage, budget, weights, ladder=False, depth=True):
    M = N + 1
    p = prep(N)
    dc = depth_cost(M) if depth else 0.0
    n = len(p)
    R = len(weights)
    w = np.array(weights, float)
    bpm = p["bpm_N"].to_numpy(float)
    sl = p["salary_next"].to_numpy(float)
    usg = p["usg"].to_numpy(float)
    c = -np.concatenate([bpm * wr for wr in w]) / 48
    zero = np.zeros(n)
    one = np.ones(n)
    rows = []
    lo = []
    hi = []

    def add(parts, low, high):
        rows.append(np.concatenate(parts))
        lo.append(low)
        hi.append(high)

    for k in range(n):
        e = np.zeros(n)
        e[k] = 1
        add([e] * R, -np.inf, 1)
    for ps in POS:
        m = (p["pos1"] == ps).to_numpy(float)
        if ladder:
            add([m] * R, 2, 2)
        else:
            for r in range(R):
                parts = [zero] * R
                parts[r] = m
                add(parts, 1, 1)
    if ladder:
        for r in range(R):
            parts = [zero] * R
            parts[r] = one
            add(parts, 1, 1)
    add([sl] * R, -np.inf, budget - dc)
    limits = [("allstar_N", 2), ("rookie_scale", 3), ("max_like", 2)]
    for col, lim in limits:
        v = p[col].to_numpy(float)
        add([v] * R, -np.inf, lim)
    if use_usage:
        add([usg * wr for wr in w], -np.inf, 100 * 48)

    A = np.vstack(rows)
    cons = LinearConstraint(A, np.array(lo), np.array(hi))
    res = milp(c, constraints=cons, integrality=np.ones(R * n),
               bounds=Bounds(0, 1))
    if res.x is None:
        print("no solution:", res.message)
        return None
    x = np.round(res.x).astype(int).reshape(R, n)
    pred = 0.0
    real = 0.0
    pay = 0.0
    use = 0.0
    for r in range(R):
        for k in np.where(x[r] == 1)[0]:
            pred += bpm[k] * w[r]
            real += p.at[k, "bpm_real"] * w[r]
            pay += sl[k]
            use += usg[k] * w[r]
    return {"pred": pred / 48, "real": real / 48,
            "payroll": pay + dc, "usage": use / 48}


def real_template(M, weights, ladder):
    if ladder:
        w = np.array(weights, float)
    else:
        w = np.repeat(np.array(weights, float), 5)
    t = adv[(adv["lg"] == "NBA") & (adv["season"] == M)]
    t = t[t["team"] != "TOT"].dropna(subset=["bpm", "mp"])
    out = {}
    for tm, g in t.groupby("team"):
        g = g.sort_values("mp", ascending=False).head(len(w))
        b = g["bpm"].to_numpy()
        out[tm] = float((b * w[: len(g)]).sum() / 48)
    return pd.Series(out)


LADDER = [36, 34, 32, 30, 28, 24, 20, 16, 12, 8]
VARS = [
    ("36/12", [36, 12], False, True),
    ("34/14", [34, 14], False, True),
    ("32/16", [32, 16], False, True),
    ("ladder", LADDER, True, True),
    ("30/14/4 (15 players)", [30, 14, 4], False, False),
]

for N in (2024, 2025):
    M = N + 1
    print("=== pair", N, "->", M, "(usage limit on) ===")
    for name, wts, lad, dep in VARS:
        rt = real_template(M, wts, lad)
        r = solve_gen(N, True, CAP[M], wts, lad, dep)
        if r is None:
            continue
        print(name, "| optimizer:", round(r["real"], 1),
              "(expected", round(r["pred"], 1), ")",
              "| real teams: median", round(rt.median(), 1),
              ", best", round(rt.max(), 1))
    r = solve_gen(N, True, TAX[M], [36, 12], False, True)
    print("budget up to the tax line, 36/12:", round(r["real"], 1),
          "(expected", round(r["pred"], 1), "), payroll, M:",
          round(r["payroll"] / 1e6, 1))
    chk = solve_gen(N, False, CAP[M], [36, 12], False, True)
    print("check: 36/12 without usage limit (earlier run: 20.3 / 19.3):",
          round(chk["real"], 1))
    print()

=== pair 2024 -> 2025 (usage limit on) ===
36/12 | optimizer: 24.1 (expected 23.8 ) | real teams: median -0.1 , best 16.2
34/14 | optimizer: 22.6 (expected 23.0 ) | real teams: median -0.0 , best 16.0
32/16 | optimizer: 21.8 (expected 22.4 ) | real teams: median 0.1 , best 15.7
ladder | optimizer: 23.8 (expected 24.9 ) | real teams: median 0.6 , best 16.4
30/14/4 (15 players) | optimizer: 20.9 (expected 21.3 ) | real teams: median -0.3 , best 14.4
budget up to the tax line, 36/12: 27.0 (expected 25.3 ), payroll, M: 169.7
check: 36/12 without usage limit (earlier run: 20.3 / 19.3): 20.3

=== pair 2025 -> 2026 (usage limit on) ===
36/12 | optimizer: 22.3 (expected 27.6 ) | real teams: median 2.8 , best 15.6
34/14 | optimizer: 20.3 (expected 26.7 ) | real teams: median 2.5 , best 15.3
32/16 | optimizer: 17.5 (expected 26.1 ) | real teams: median 2.2 , best 15.0
ladder | optimizer: 22.8 (expected 27.9 ) | real teams: median 2.9 , best 15.5
30/14/4 (15 players) | optimizer: 17.5 (expected 2

## Notes and limitations

- BPM is not perfectly additive. Players share the ball and the court, and the usage constraint only partly corrects this.
- Every player is assumed to be healthy and to play fixed minutes for the whole season. Real stars miss games.
- The squad is picked from all players in the league, regardless of whether they could be acquired.
- Ratings of season N are noisy, and the selection picks lucky seasons, so expected values are higher than realised ones.
- Two season pairs only. Cap and tax-line values for 2025 were entered by hand.
- Rookie-scale contracts are approximated from draft round and years in the league.